# 07 · Go deeper (web)
Opt-in only. The student presses **Go deeper**; the app must tell them that the question leaves the computer. A fixed pipeline, not an agent loop:

search (Wikipedia API + ddgs) → prefer allowlisted sites → fetch the top 4 pages (trafilatura) → chunk, embed, retrieve → answer citing `[W#]` with URLs → **confirmed by 2 sources** when sentences are backed by two independent sites.

Shown in a separate "Beyond your materials" section, never mixed with the grounded answer. "Add to my sources" stores the web chunks with `role=web`. Without a network the model's own knowledge is used, labelled **AI-generated, not verified**.

Pages are cached in `data/web_cache/` so a re-run doesn't fetch them again. Output: `data/eval/web_results.json`.

## 1 · Load the 05 pipeline
Same approach as 06: 05's code cells up to its Checks section are run here (LLM call, citation helpers, support check, embedding settings).

In [1]:
import hashlib, json, re, time
from urllib.parse import urlparse

import nbformat
import trafilatura
from ddgs import DDGS


def run_pipeline_cells(path="05_generation.ipynb"):
    nb = nbformat.read(path, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == "markdown" and cell.source.lstrip().startswith("---") and "# Checks" in cell.source:
            break
        if cell.cell_type == "code":
            exec(cell.source, globals())


run_pipeline_cells()
print(f"from 05: LLM {LLM} | embedder {EMBED_MODEL}")

f:\RP\Edu-Rag\rp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 358.11it/s]


retrieval: dense | bge-m3 | reorder=False | dense floor 0.45 | old reranker threshold 0.261 (comparison only) | 495 chunks
from 05: LLM qwen2.5:7b-instruct | embedder bge-m3


## 2 · Settings

In [2]:
WEB_CACHE = DATA_DIR / "web_cache"
WEB_CACHE.mkdir(parents=True, exist_ok=True)
USER_AGENT = "EduRAG-student-project/0.1 (local study assistant)"
ALLOWLIST = ["docs.python.org", "wikipedia.org", "realpython.com", "geeksforgeeks.org", "w3schools.com",
             "programiz.com", "numpy.org", "pandas.pydata.org"]
DDGS_BACKENDS = ["duckduckgo", "bing", "google"]   # tried in turn; yahoo failed its TLS handshake on this network
N_SEARCH = 8
N_PAGES = 4
MAX_PAGE_CHARS = 20000                          # keep the first part of long pages
WEB_K = 5
CONFIRM_MIN_OVERLAP = SUPPORT_MIN               # a cited passage must back the sentence to count as agreeing
NOTICE = "Go deeper sends your question to web search (Wikipedia, DuckDuckGo) and downloads the top pages. It leaves your computer."
OFFLINE_LABEL = "AI-generated, not verified"
SENT_LOG = []                                   # every query that left the machine, for the privacy check

## 3 · Search
Wikipedia's search API and ddgs, merged and de-duplicated. Allowlisted sites first, then the rest. At most one page per site, so the 4 pages come from independent sites where possible.

In [3]:
def domain_of(url):
    host = urlparse(url).netloc.lower()
    return host[4:] if host.startswith("www.") else host


def allowlisted(url):
    d = domain_of(url)
    return any(d == a or d.endswith("." + a) for a in ALLOWLIST)


def site_key(url):
    """Independent-site key: en.wikipedia.org and de.wikipedia.org are one site."""
    d = domain_of(url)
    return next((a for a in ALLOWLIST if d == a or d.endswith("." + a)), d)


QUESTION_WORDS = set("how do does did i can could should would what which why when explain tell me about way ways "
                     "use using difference between".split())


def keywords(query):
    """Wikipedia's search matches all words, so a full question finds odd pages ('Python set' -> Monty Python)."""
    return " ".join(w for w in tokenize(query) if w not in QUESTION_WORDS)


def wikipedia_search(query, n=3):
    r = requests.get("https://en.wikipedia.org/w/api.php", timeout=15, headers={"User-Agent": USER_AGENT},
                     params=dict(action="query", list="search", srsearch=keywords(query), format="json", srlimit=n))
    r.raise_for_status()
    return [dict(url="https://en.wikipedia.org/wiki/" + s["title"].replace(" ", "_"), title=s["title"], engine="wikipedia")
            for s in r.json()["query"]["search"]]


def ddgs_search(query, n=N_SEARCH):
    last = None
    for backend in DDGS_BACKENDS:
        try:
            res = DDGS(timeout=10).text(query, max_results=n, backend=backend)
            if res:
                return [dict(url=x["href"], title=x.get("title", ""), engine=f"ddgs/{backend}") for x in res]
        except Exception as e:
            last = e
    raise RuntimeError(f"no results from {DDGS_BACKENDS}: {str(last)[:60]}")


def web_search(query):
    SENT_LOG.append(query)
    hits, errors = [], []
    for fn in (wikipedia_search, ddgs_search):
        try:
            hits += fn(query)
        except Exception as e:
            errors.append(f"{fn.__name__}: {str(e)[:80]}")
    seen, uniq = set(), []
    for h in hits:
        u = h["url"].split("#")[0]
        if u not in seen:
            seen.add(u)
            uniq.append(dict(h, url=u))
    ranked = sorted(uniq, key=lambda h: not allowlisted(h["url"]))           # stable: keeps engine order inside groups
    picked, sites = [], set()
    for h in ranked:
        if site_key(h["url"]) not in sites:
            picked.append(h)
            sites.add(site_key(h["url"]))
    return picked, errors

## 4 · Fetch and extract (cached)

In [4]:
def fetch_page(url):
    path = WEB_CACHE / (hashlib.sha1(url.encode()).hexdigest()[:16] + ".json")
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8")) | {"cached": True}
    try:
        r = requests.get(url, timeout=15, headers={"User-Agent": USER_AGENT})
        r.raise_for_status()
        text = trafilatura.extract(r.text, include_tables=True, include_comments=False, favor_precision=True) or ""
        meta = trafilatura.extract_metadata(r.text)
        page = dict(url=url, title=(meta.title if meta and meta.title else url), text=text[:MAX_PAGE_CHARS], ok=bool(text))
    except Exception as e:
        page = dict(url=url, title=url, text="", ok=False, error=str(e)[:120])
    if page["ok"]:
        path.write_text(json.dumps(page, ensure_ascii=False), encoding="utf-8")
    return page | {"cached": False}


def fetch_top(hits, n=N_PAGES):
    """Fetch in ranked order until n pages gave usable text."""
    pages = []
    for h in hits:
        p = fetch_page(h["url"])
        if p["ok"] and len(p["text"]) > 300:
            pages.append(p)
        if len(pages) == n:
            break
    return pages

## 5 · Chunk, embed, retrieve
Paragraphs packed up to the same 400-token limit as 02, each chunk prefixed with the page title. Embedded with the same model as the course chunks, searched in memory.

In [5]:
WEB_MAX_TOKENS = 400


def chunk_page(page):
    paras = [p.strip() for p in page["text"].split("\n") if len(p.strip()) > 40]
    chunks_, cur = [], []
    budget = WEB_MAX_TOKENS - round(len(page["title"]) / 4) - 2
    for p in paras:
        if cur and round(len("\n".join(cur + [p])) / 4) > budget:
            chunks_.append(cur)
            cur = []
        cur.append(p[: budget * 4])
    if cur:
        chunks_.append(cur)
    sid = "web-" + hashlib.sha1(page["url"].encode()).hexdigest()[:10]
    return [dict(chunk_id=f"{sid}-{i:03d}", source_id=sid, source_name=f"{domain_of(page['url'])}: {page['title']}",
                 role="web", url=page["url"], domain=domain_of(page["url"]), site=site_key(page["url"]),
                 page_start=0, page_end=0, title=page["title"], has_code="```" in "\n".join(c), has_table=False,
                 text=f"{page['title']}\n\n" + "\n".join(c)) for i, c in enumerate(chunks_)]


def embed_docs(texts):
    out = []
    for i in range(0, len(texts), 16):
        r = requests.post(f"{OLLAMA_URL}/api/embed", timeout=300, json={"model": EMBED_MODEL, "keep_alive": "10m",
                          "input": [EMBEDDERS[EMBED_MODEL]["doc_prefix"] + t for t in texts[i:i + 16]]})
        r.raise_for_status()
        out += r.json()["embeddings"]
    v = np.asarray(out, dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


def web_retrieve(query, web_chunks, k=WEB_K):
    if not web_chunks:
        return [], [], None
    vecs = embed_docs([c["text"] for c in web_chunks])
    sims = vecs @ embed_query(query)
    order = np.argsort(-sims)[:k]
    return [web_chunks[i] for i in order], [float(sims[i]) for i in order], vecs

## 6 · Answer with `[W#]` citations
Same rules as the grounded answer, but the passages are web pages labelled `[W1]..[Wn]`. Code checks the citations, attaches the URLs, and runs the support check.

In [6]:
WEB_SYSTEM_PROMPT = SYSTEM_PROMPT.replace("[S1]", "[W1]").replace("[S2][S3]", "[W2][W3]").replace("[S2]", "[W2]") \
    .replace("numbered passages from their course materials", "numbered passages from web pages")
WEB_CITE = re.compile(r"\[\s*W\d+(?:\s*(?:,|-|–)\s*W?\d+)*\s*\]")


def web_messages(question, passages, style):
    body = "\n\n".join(f"[W{i}] ({c['domain']}, {c['title']})\n{c['text']}" for i, c in enumerate(passages, start=1))
    return [{"role": "system", "content": WEB_SYSTEM_PROMPT},
            {"role": "user", "content": f"Passages:\n{body}\n\nStyle: {STYLES[style]}\n\nQuestion: {question}"}]


def parse_web_citations(text, n):
    """Reuse the [S#] parser by translating W <-> S."""
    as_s = WEB_CITE.sub(lambda m: m.group(0).replace("W", "S"), text)
    cleaned, valid, invalid = parse_citations(as_s, n)
    return CITE_GROUP.sub(lambda m: m.group(0).replace("S", "W"), cleaned), valid, invalid


def confirmed_sentences(text, passages):
    """Sentences whose cited passages back them (overlap >= CONFIRM_MIN_OVERLAP) from at least two independent sites."""
    as_s = WEB_CITE.sub(lambda m: m.group(0).replace("W", "S"), text)
    out = []
    for s in split_sentences(as_s):
        nums = sorted({x for g in CITE_GROUP.findall(s) for x in cite_numbers(g) if 1 <= x <= len(passages)})
        words = content_words(CITE_GROUP.sub("", s))
        backing = {passages[x - 1]["site"] for x in nums
                   if words and len(words & content_words(passages[x - 1]["text"])) / len(words) >= CONFIRM_MIN_OVERLAP}
        out.append(dict(sentence=CITE_GROUP.sub(lambda m: m.group(0).replace("S", "W"), s), sites=sorted(backing),
                        confirmed=len(backing) >= 2))
    return out


def offline_answer(question, style):
    msgs = [{"role": "system", "content": "You are a study assistant. The student's course materials don't cover this and the web is "
                                          "unavailable. Give a short general explanation from your own knowledge. Say nothing about sources."},
            {"role": "user", "content": f"Style: {STYLES[style]}\n\nQuestion: {question}"}]
    text, stats = chat(msgs)
    return text, stats

## 7 · The Go deeper pipeline

In [7]:
def go_deeper(question, style="concise", offline=False):
    t0 = time.time()
    res = dict(section="Beyond your materials", notice=NOTICE, question=question, mode="web", label=None, answer=None,
               sources=[], confirmed=[], badge=False, search_errors=[], pages=[], citations_ok=False)
    hits, errors = ([], ["offline"]) if offline else web_search(question)
    res["search_errors"] = errors
    pages = fetch_top(hits) if hits else []
    res["pages"] = [dict(url=p["url"], title=p["title"], chars=len(p["text"]), cached=p["cached"]) for p in pages]
    if not pages:
        text, stats = offline_answer(question, style)
        res.update(mode="offline", label=OFFLINE_LABEL, answer=text, llm=stats, seconds=round(time.time() - t0, 2))
        return res
    web_chunks = [c for p in pages for c in chunk_page(p)]
    passages, sims, vecs = web_retrieve(question, web_chunks)
    raw, stats = chat(web_messages(question, passages, style))
    res.update(llm=stats, web_chunks=web_chunks, web_vectors=vecs, top_sim=sims[0] if sims else 0.0)
    if INSUFFICIENT in raw:
        text, stats2 = offline_answer(question, style)
        res.update(mode="offline", label=OFFLINE_LABEL + " (the fetched pages did not cover it)", answer=text)
    else:
        text, valid, invalid = parse_web_citations(raw, len(passages))
        conf = confirmed_sentences(text, passages)
        res.update(answer=text, invalid_citations=invalid, citations_ok=bool(valid), confirmed=conf,
                   badge=any(c["confirmed"] for c in conf),
                   sources=[dict(label=f"W{n}", url=passages[n - 1]["url"], domain=passages[n - 1]["domain"],
                                 title=passages[n - 1]["title"], allowlisted=allowlisted(passages[n - 1]["url"])) for n in valid])
    res["seconds"] = round(time.time() - t0, 2)
    return res


def show_deeper(res):
    print(f"== {res['section']} ==   ({res['notice'][:60]}...)")
    print(f"Q: {res['question']}   [{res['mode']}, {res['seconds']} s, pages {len(res['pages'])}]")
    if res["label"]:
        print(f"[{res['label']}]")
    if res["badge"]:
        n = sum(c["confirmed"] for c in res["confirmed"])
        print(f"[confirmed by 2 sources: {n} of {len(res['confirmed'])} sentences]")
    print(res["answer"])
    for s in group_sources(res["sources"]):
        print(f"   [{', '.join(s['labels'])}] {s['title'][:50]} - {s['url']}" + ("" if s["allowlisted"] else "  (not on allowlist)"))
    print()


def group_sources(sources):
    """Several [W#] passages can come from one page; list each page once with all its labels."""
    by_url = {}
    for s in sources:
        by_url.setdefault(s["url"], dict(s, labels=[]))["labels"].append(s["label"])
    return list(by_url.values())


def unload_models():
    for m in (LLM, EMBED_MODEL):
        requests.post(f"{OLLAMA_URL}/api/generate", json={"model": m, "keep_alive": 0}, timeout=60)
    for _ in range(10):
        if not requests.get(f"{OLLAMA_URL}/api/ps").json()["models"]:
            break
        time.sleep(0.5)

## 8 · Add to my sources
Turns the fetched web chunks into normal `role=web` chunks: added to a Chroma collection with their vectors, so later grounded answers can cite them. The demo below uses a temporary in-memory collection, so the real index in `data/chroma` is not changed by this notebook.

In [8]:
def add_to_sources(res, collection):
    chunks_ = res.get("web_chunks") or []
    if not chunks_:
        return 0
    keys = ["chunk_id", "source_id", "source_name", "role", "page_start", "page_end", "title", "has_code", "has_table", "url"]
    collection.upsert(ids=[c["chunk_id"] for c in chunks_], embeddings=res["web_vectors"].tolist(),
                      documents=[c["text"] for c in chunks_], metadatas=[{k: c[k] for k in keys} for c in chunks_])
    return len(chunks_)

---
# Checks

## 9 · Search ranking for three questions

In [9]:
for q in ["What does dropout do when training a neural network?", "How do I add and remove items from a Python set?",
          "How does a hidden Markov model work?"]:
    hits, errs = web_search(q)
    print(f"Q: {q}   errors: {errs or 'none'}")
    for h in hits[:6]:
        print(f"   {'ALLOW' if allowlisted(h['url']) else '     '} {h['engine']:9} {h['url'][:90]}")

Q: What does dropout do when training a neural network?   errors: none
   ALLOW wikipedia https://en.wikipedia.org/wiki/Dropout_(neural_networks)
   ALLOW ddgs/bing https://www.geeksforgeeks.org/machine-learning/dropout-in-neural-networks/
         ddgs/bing https://towardsdatascience.com/dropout-in-neural-networks-47a162d621d9/
         ddgs/bing https://mlforbeginners.com/dropout/
         ddgs/bing https://medium.com/biased-algorithms/the-role-of-dropout-in-neural-networks-fffbaa77eee7
         ddgs/bing https://www.coursera.org/articles/dropout-neural-network
Q: How do I add and remove items from a Python set?   errors: none
   ALLOW wikipedia https://en.wikipedia.org/wiki/Monty_Python
   ALLOW ddgs/bing https://www.geeksforgeeks.org/python/python-set-methods/
   ALLOW ddgs/bing https://www.w3schools.com/python/python_sets_add.asp
         ddgs/bing https://pytutorial.com/python-set-methods-guide-add-remove-compare/
         ddgs/bing https://rune.codes/hub/python/add-and-remove-se

## 10 · Go deeper on the eval set's unanswerable questions
These are exactly the questions the course materials don't cover, so this is where Go deeper is meant to be used.

In [10]:
with open(EVAL_DIR / "questions.csv", encoding="utf-8") as f:
    beyond = [r["question"] for r in csv.DictReader(f) if r["answerable"].strip().lower() == "no"]
t0 = time.time()
web_runs = [go_deeper(q) for q in beyond]
print(f"{len(web_runs)} questions in {time.time() - t0:.0f} s\n")
for r in web_runs[:4]:
    show_deeper(r)

16 questions in 269 s

== Beyond your materials ==   (Go deeper sends your question to web search (Wikipedia, Duck...)
Q: What is the Apriori algorithm?   [web, 16.17 s, pages 4]
[confirmed by 2 sources: 1 of 3 sentences]
The Apriori algorithm is an unsupervised machine learning method used for association rule learning. It identifies frequent patterns and dependencies among itemsets in data, such as in market basket analysis or fraud detection [W1][W2]. The algorithm works by generating candidate item sets and pruning those that are infrequent, leveraging the downward closure property to efficiently find frequent item sets [W3][W4].
   [W1, W3] Apriori algorithm - Wikipedia - https://en.wikipedia.org/wiki/Apriori_algorithm
   [W2, W4] What is the Apriori algorithm? | IBM - https://www.ibm.com/think/topics/apriori-algorithm  (not on allowlist)

== Beyond your materials ==   (Go deeper sends your question to web search (Wikipedia, Duck...)
Q: What is a fuzzy membership function?   [web,

## 11 · Numbers

In [11]:
web = [r for r in web_runs if r["mode"] == "web"]
srcs = [s for r in web for s in r["sources"]]
conf = [c for r in web for c in r["confirmed"]]
print(f"answered from the web: {len(web)}/{len(web_runs)} | offline fallback: {sum(r['mode'] == 'offline' for r in web_runs)}")
print(f"answers with valid [W#] citations: {sum(r['citations_ok'] for r in web)}/{len(web)} | invalid citation numbers: {sum(len(r.get('invalid_citations', [])) for r in web)}")
pages_cited = [s for r in web for s in group_sources(r["sources"])]
print(f"cited pages on the allowlist: {sum(s['allowlisted'] for s in pages_cited)}/{len(pages_cited)} | distinct sites cited per answer: "
      f"{np.mean([len({s['site'] if 'site' in s else site_key(s['url']) for s in r['sources']}) for r in web]):.1f}")
print("Wikipedia keyword queries, e.g.:", [keywords(q) for q in beyond[:3]])
print(f"'confirmed by 2 sources' badge: {sum(r['badge'] for r in web)}/{len(web)} answers; confirmed sentences {sum(c['confirmed'] for c in conf)}/{len(conf)}")
print(f"pages fetched per question: {np.mean([len(r['pages']) for r in web_runs]):.1f} | search errors: "
      f"{Counter(e.split(':')[0] for r in web_runs for e in r['search_errors'])}")
secs = [r["seconds"] for r in web_runs]
print(f"time: mean {np.mean(secs):.1f} s, max {max(secs):.1f} s (first run includes fetching; cached pages are faster)")

answered from the web: 16/16 | offline fallback: 0
answers with valid [W#] citations: 16/16 | invalid citation numbers: 0
cited pages on the allowlist: 16/38 | distinct sites cited per answer: 2.4
Wikipedia keyword queries, e.g.: ['apriori algorithm', 'fuzzy membership function', 'crossover operator genetic algorithm']
'confirmed by 2 sources' badge: 9/16 answers; confirmed sentences 10/37
pages fetched per question: 4.0 | search errors: Counter()
time: mean 16.8 s, max 26.6 s (first run includes fetching; cached pages are faster)


## 12 · Offline fallback

In [12]:
show_deeper(go_deeper("What does dropout do when training a neural network?", offline=True))

== Beyond your materials ==   (Go deeper sends your question to web search (Wikipedia, Duck...)
Q: What does dropout do when training a neural network?   [offline, 2.9 s, pages 0]
[AI-generated, not verified]
During training, dropout randomly selects neurons and temporarily removes them from the network, which helps prevent overfitting by making the model more robust and generalizable. This process is akin to training multiple networks with different architectures and acts as a form of regularization.



## 13 · Add to my sources (temporary collection)

In [13]:
demo_col = chromadb.EphemeralClient().get_or_create_collection("web_demo", metadata={"hnsw:space": "cosine"})
r = next(x for x in web_runs if x["mode"] == "web")
n = add_to_sources(r, demo_col)
hit = demo_col.query(query_embeddings=[embed_query(r["question"]).tolist()], n_results=2, where={"role": "web"}, include=["metadatas"])
print(f"added {n} chunks with role=web from: {r['question']}")
for m in hit["metadatas"][0]:
    print("   retrievable as evidence:", m["source_name"][:60], "-", m["url"])
print("real index untouched:", col.count(), "chunks in", col.name)

added 24 chunks with role=web from: What is the Apriori algorithm?
   retrievable as evidence: en.wikipedia.org: Apriori algorithm - Wikipedia - https://en.wikipedia.org/wiki/Apriori_algorithm
   retrievable as evidence: ibm.com: What is the Apriori algorithm? | IBM - https://www.ibm.com/think/topics/apriori-algorithm
real index untouched: 495 chunks in chunks_bge_m3


## 14 · Privacy check: everything that left the machine

In [14]:
print(f"{len(SENT_LOG)} search queries sent; each is only the question text:")
for q in SENT_LOG:
    print("  ", q)

19 search queries sent; each is only the question text:
   What does dropout do when training a neural network?
   How do I add and remove items from a Python set?
   How does a hidden Markov model work?
   What is the Apriori algorithm?
   What is a fuzzy membership function?
   What is the crossover operator in a genetic algorithm?
   How does a convolutional neural network work?
   How does backpropagation compute gradients?
   How does the attention mechanism in transformers work?
   How do I create a class that inherits from another class in Python?
   How do I connect to a MySQL database from Python?
   What is the difference between TCP and UDP?
   What is the difference between a primary key and a foreign key?
   How does a hidden Markov model work?
   How should I choose the perplexity value for t-SNE?
   What does dropout do when training a neural network?
   How does momentum speed up gradient descent?
   How do I add and remove items from a Python set?
   How does gradient 

## 15 · Save

In [ ]:
keep = ["question", "mode", "label", "answer", "sources", "confirmed", "badge", "pages", "search_errors", "citations_ok", "seconds"]
(EVAL_DIR / "web_results.json").write_text(json.dumps(dict(
    allowlist=ALLOWLIST, n_pages=N_PAGES, runs=[{k: r.get(k) for k in keep} for r in web_runs]), indent=2, ensure_ascii=False))
unload_models()
print("saved web_results.json |", len(list(WEB_CACHE.glob("*.json"))), "pages cached |",
      [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/ps").json()["models"]] or "no Ollama models loaded")

saved web_results.json | 95 pages cached | no Ollama models loaded


: 